# META-CXR — Evaluation & Results Visualization

Notebook này đọc kết quả training từ dataset checkpoint và visualize:
- **Training loss curves** (total + từng component)
- **Validation metrics** (BLEU-1/2/3/4, METEOR, ROUGE-L, Agg)
- **Sample predictions** vs ground truth
- **Checkpoint summary**

**Prerequisites:**
- Attach dataset `meta-cxr-checkpoints` (từ Notebook Settings → Add Data)
- Hoặc chạy trong cùng session sau khi training xong

**Steps:** Chạy Cell 1 → 2 → 3 → 4 → 5 → 6 theo thứ tự.

## Cell 1 — Setup

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, HTML

REPO_DIR = "/kaggle/working/META-CXR"

# Clone or update repo (cần configs/kaggle_datasets.yaml)
if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull -q")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

# Load dataset config
import yaml
with open("configs/kaggle_datasets.yaml") as f:
    CFG = yaml.safe_load(f)

# Matplotlib style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.color": "white",
    "grid.linewidth": 1.2,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Setup OK.")

## Cell 2 — Locate Results

In [ ]:
def find_mount(slug):
    for root in CFG["mount_search_roots"]:
        candidate = os.path.join(root, slug)
        if os.path.isdir(candidate):
            return candidate
    matches = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    return matches[0] if matches else None

CKPT_SLUG  = CFG["datasets"]["checkpoints"]["slug"]
LOCAL_OUT  = "/kaggle/working/output"

# Priority: local output dir (same session) > attached dataset
ckpt_root = find_mount(CKPT_SLUG)

if os.path.isdir(LOCAL_OUT) and glob.glob(f"{LOCAL_OUT}/**/log.txt", recursive=True):
    OUTPUT_DIR = LOCAL_OUT
    print(f"Using local output: {OUTPUT_DIR}")
elif ckpt_root:
    OUTPUT_DIR = ckpt_root
    print(f"Using attached dataset: {OUTPUT_DIR}")
else:
    raise FileNotFoundError(
        f"No results found.\n"
        f"  Option A: Run this notebook in the same session as training.\n"
        f"  Option B: Attach dataset '{CKPT_SLUG}' via Notebook Settings → Add Data."
    )

# Find log file
log_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/log.txt", recursive=True))
print(f"\nFound {len(log_files)} log file(s):")
for lf in log_files:
    print(f"  {lf}")

# Parse all JSON lines from all logs
all_records = []
for lf in log_files:
    with open(lf) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("{\""):
                continue
            try:
                all_records.append(json.loads(line))
            except Exception:
                pass

# Separate train vs val records
train_records, val_records, loss_records = [], [], []
for r in all_records:
    keys = set(r.keys())
    if any(k.startswith("train_") for k in keys):
        train_records.append(r)
    elif any(k.startswith("val_") and k not in ("val_loss",) for k in keys):
        val_records.append(r)
    elif "val_loss" in keys:
        loss_records.append(r)

print(f"\nParsed: {len(train_records)} train records, {len(val_records)} val metric records, {len(loss_records)} val loss records")

## Cell 3 — Training Loss Curves

In [ ]:
if not train_records:
    print("No training records found.")
else:
    df_train = pd.DataFrame(train_records)
    # Assign epoch from column or infer from row order
    if "train_epoch" in df_train.columns:
        df_train["epoch"] = df_train["train_epoch"]
    else:
        df_train["epoch"] = range(len(df_train))

    loss_cols = [c for c in df_train.columns if "loss" in c.lower() and c != "train_epoch"]
    lr_cols   = [c for c in df_train.columns if "lr" in c.lower()]

    n_loss = len(loss_cols)
    has_lr = len(lr_cols) > 0

    fig = plt.figure(figsize=(16, 5 if not has_lr else 10))
    gs  = gridspec.GridSpec(2 if has_lr else 1, 1, hspace=0.4)

    # --- Loss subplot ---
    ax_loss = fig.add_subplot(gs[0])
    colors = plt.cm.tab10.colors
    for i, col in enumerate(loss_cols):
        label = col.replace("train_", "").replace("_", " ").title()
        lw = 2.5 if col == "train_loss" else 1.2
        ls = "-" if col == "train_loss" else "--"
        ax_loss.plot(df_train["epoch"], df_train[col], label=label,
                     color=colors[i % 10], linewidth=lw, linestyle=ls, marker="o", markersize=4)
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.set_title("Training Loss", fontweight="bold", fontsize=13)
    ax_loss.legend(loc="upper right", fontsize=9, ncol=2)
    ax_loss.set_xticks(df_train["epoch"])

    # --- LR subplot ---
    if has_lr:
        ax_lr = fig.add_subplot(gs[1])
        for col in lr_cols:
            ax_lr.plot(df_train["epoch"], df_train[col], color="steelblue",
                       linewidth=2, marker="o", markersize=4)
        ax_lr.set_xlabel("Epoch")
        ax_lr.set_ylabel("Learning Rate")
        ax_lr.set_title("Learning Rate Schedule", fontweight="bold", fontsize=13)
        ax_lr.set_xticks(df_train["epoch"])
        ax_lr.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

    plt.suptitle("Training Curves", fontsize=15, fontweight="bold", y=1.01)
    plt.savefig("/kaggle/working/training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nTraining summary:")
    display(df_train[["epoch"] + loss_cols[:5]].set_index("epoch").round(4))

## Cell 4 — Validation Metrics

In [ ]:
if not val_records and not loss_records:
    print("No validation records found.")
else:
    # --- Val metrics (BLEU, METEOR, ROUGE, agg) ---
    if val_records:
        df_val = pd.DataFrame(val_records)
        df_val["epoch"] = range(len(df_val))
        metric_cols = [c for c in df_val.columns
                       if c.startswith("val_") and "best_epoch" not in c and "epoch" not in c]

        # Group metrics
        bleu_cols  = [c for c in metric_cols if "BLEU" in c or "Bleu" in c]
        other_cols = [c for c in metric_cols if c not in bleu_cols]

        n_panels = (1 if bleu_cols else 0) + len(other_cols)
        fig, axes = plt.subplots(1, min(n_panels, 4), figsize=(5 * min(n_panels, 4), 4.5))
        if n_panels == 1:
            axes = [axes]

        panel = 0
        colors = plt.cm.Set2.colors

        # BLEU panel
        if bleu_cols and panel < len(axes):
            ax = axes[panel]; panel += 1
            for i, col in enumerate(bleu_cols):
                label = col.replace("val_", "").replace("_", "-")
                ax.plot(df_val["epoch"], df_val[col], label=label,
                        color=colors[i % len(colors)], linewidth=2, marker="o", markersize=5)
            ax.set_title("BLEU Scores", fontweight="bold")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
            ax.legend(fontsize=9); ax.set_xticks(df_val["epoch"])

        # Other metrics (METEOR, ROUGE, agg)
        for col in other_cols:
            if panel >= len(axes):
                break
            ax = axes[panel]; panel += 1
            label = col.replace("val_", "").replace("_", " ")
            color = "#2196F3" if "agg" in col else "#4CAF50" if "ROUGE" in col else "#FF9800"
            ax.plot(df_val["epoch"], df_val[col], color=color, linewidth=2.5, marker="o", markersize=6)
            # Mark best
            best_idx = df_val[col].idxmax()
            ax.scatter(df_val.loc[best_idx, "epoch"], df_val.loc[best_idx, col],
                       s=120, color="gold", edgecolor="black", zorder=5, label=f"Best: {df_val.loc[best_idx, col]:.4f}")
            ax.set_title(label, fontweight="bold")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
            ax.legend(fontsize=9); ax.set_xticks(df_val["epoch"])

        plt.suptitle("Validation Metrics", fontsize=15, fontweight="bold")
        plt.tight_layout()
        plt.savefig("/kaggle/working/val_metrics.png", dpi=150, bbox_inches="tight")
        plt.show()

        print("\nValidation summary (best per metric):")
        summary = {}
        for col in metric_cols:
            if col in df_val.columns:
                best_val = df_val[col].max()
                best_ep  = df_val.loc[df_val[col].idxmax(), "epoch"]
                summary[col.replace("val_", "")] = {"best": round(best_val, 4), "at epoch": int(best_ep)}
        display(pd.DataFrame(summary).T)

    # --- Val loss ---
    if loss_records:
        df_loss = pd.DataFrame(loss_records)
        df_loss["epoch"] = range(len(df_loss))
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(df_loss["epoch"], df_loss["val_loss"], color="#E91E63",
                linewidth=2.5, marker="o", markersize=6)
        best_idx = df_loss["val_loss"].idxmin()
        ax.scatter(df_loss.loc[best_idx, "epoch"], df_loss.loc[best_idx, "val_loss"],
                   s=150, color="gold", edgecolor="black", zorder=5,
                   label=f"Best: {df_loss.loc[best_idx, 'val_loss']:.4f}")
        ax.set_title("Validation Loss", fontweight="bold", fontsize=13)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.set_xticks(df_loss["epoch"])
        plt.tight_layout()
        plt.savefig("/kaggle/working/val_loss.png", dpi=150, bbox_inches="tight")
        plt.show()

## Cell 5 — Sample Predictions

In [ ]:
# Find prediction + ground truth files
pred_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/predictions_*.txt", recursive=True))
gt_files   = sorted(glob.glob(f"{OUTPUT_DIR}/**/ground_truths_*.txt", recursive=True))

if not pred_files:
    print("No prediction files found.\nPredictions are saved during evaluate_only mode or after val epoch.")
else:
    def read_lines(path):
        with open(path) as f:
            lines = [l.strip().strip('"') for l in f if l.strip()]
        return lines

    for pred_f in pred_files[:2]:
        split = os.path.basename(pred_f).replace("predictions_", "").replace(".txt", "")
        gt_f  = next((g for g in gt_files if split in g), None)

        preds = read_lines(pred_f)
        gts   = read_lines(gt_f) if gt_f else ["N/A"] * len(preds)
        n     = min(len(preds), len(gts))

        print(f"\n{'='*80}")
        print(f"Split: {split}  |  Total: {len(preds)} predictions")
        print(f"{'='*80}")

        # Table of first 10 samples
        SHOW = 10
        rows = []
        for i in range(min(SHOW, n)):
            rows.append({"#": i + 1, "Ground Truth": gts[i], "Prediction": preds[i]})

        df_pred = pd.DataFrame(rows).set_index("#")

        # HTML table with nice styling
        html = df_pred.to_html(escape=False)
        style = """
        <style>
        table { border-collapse: collapse; width: 100%; font-size: 12px; }
        th { background: #1565C0; color: white; padding: 8px 12px; text-align: left; }
        td { padding: 7px 12px; border-bottom: 1px solid #e0e0e0; vertical-align: top; }
        tr:nth-child(even) { background: #f5f5f5; }
        tr:hover { background: #e3f2fd; }
        td:first-child { color: #555; font-weight: bold; width: 40px; }
        td:nth-child(2) { color: #2e7d32; max-width: 400px; }
        td:nth-child(3) { color: #1a237e; max-width: 400px; }
        </style>
        """
        display(HTML(style + html))

        # Token length comparison
        pred_lens = [len(p.split()) for p in preds[:n]]
        gt_lens   = [len(g.split()) for g in gts[:n]]

        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        bins = range(0, max(max(pred_lens, default=1), max(gt_lens, default=1)) + 10, 5)
        axes[0].hist(gt_lens,   bins=bins, alpha=0.7, color="#4CAF50", label="Ground Truth", edgecolor="white")
        axes[0].hist(pred_lens, bins=bins, alpha=0.7, color="#2196F3", label="Prediction",   edgecolor="white")
        axes[0].set_xlabel("Token count"); axes[0].set_ylabel("Frequency")
        axes[0].set_title(f"Report Length Distribution ({split})", fontweight="bold")
        axes[0].legend()

        axes[1].scatter(gt_lens[:n], pred_lens[:n], alpha=0.5, s=20, color="#7B1FA2")
        max_len = max(max(gt_lens[:n], default=1), max(pred_lens[:n], default=1))
        axes[1].plot([0, max_len], [0, max_len], "r--", linewidth=1, label="Perfect match")
        axes[1].set_xlabel("GT length"); axes[1].set_ylabel("Pred length")
        axes[1].set_title("Predicted vs GT Length", fontweight="bold")
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(f"/kaggle/working/length_dist_{split}.png", dpi=150, bbox_inches="tight")
        plt.show()

## Cell 6 — Checkpoint Summary

In [ ]:
import torch

ckpt_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/checkpoint_*.pth", recursive=True))

if not ckpt_files:
    print("No checkpoint files found.")
else:
    rows = []
    for ckpt_path in ckpt_files:
        fname    = os.path.basename(ckpt_path)
        size_mb  = os.path.getsize(ckpt_path) / (1024 ** 2)
        is_best  = "best" in fname
        is_last  = "last" in fname
        epoch    = None
        try:
            meta = torch.load(ckpt_path, map_location="cpu", weights_only=True)
            epoch = meta.get("epoch", None)
        except Exception:
            try:
                meta = torch.load(ckpt_path, map_location="cpu")
                epoch = meta.get("epoch", None)
            except Exception:
                pass
        if epoch is None and not (is_best or is_last):
            try:
                epoch = int(fname.replace("checkpoint_", "").replace(".pth", ""))
            except Exception:
                pass
        rows.append({
            "File": fname,
            "Epoch": epoch if epoch is not None else "—",
            "Size (MB)": round(size_mb, 1),
            "Type": "★ Best" if is_best else ("Last" if is_last else "Periodic"),
            "Path": ckpt_path,
        })

    df_ckpt = pd.DataFrame(rows)
    total_gb = df_ckpt["Size (MB)"].sum() / 1024

    print(f"Total checkpoint storage: {total_gb:.2f} GB\n")

    # Styled display
    def style_row(row):
        if "Best" in str(row["Type"]):
            return ["background-color: #fff9c4; font-weight: bold"] * len(row)
        return [""] * len(row)

    display(
        df_ckpt[["File", "Epoch", "Size (MB)", "Type"]]
        .style
        .apply(style_row, axis=1)
        .set_caption("Checkpoint Summary")
        .set_table_styles([{
            "selector": "caption",
            "props": "font-size: 14px; font-weight: bold; margin-bottom: 8px;"
        }])
    )

    # Bar chart: checkpoint sizes
    fig, ax = plt.subplots(figsize=(max(6, len(ckpt_files) * 1.4), 4))
    colors_bar = ["#FFC107" if "Best" in r["Type"] else "#42A5F5" for r in rows]
    bars = ax.bar(df_ckpt["File"], df_ckpt["Size (MB)"], color=colors_bar, edgecolor="white", width=0.6)
    ax.bar_label(bars, fmt="%.0f MB", padding=3, fontsize=9)
    ax.set_ylabel("Size (MB)")
    ax.set_title("Checkpoint File Sizes", fontweight="bold", fontsize=13)
    ax.tick_params(axis="x", rotation=30)
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="#FFC107", label="Best checkpoint"),
        Patch(color="#42A5F5", label="Periodic checkpoint"),
    ], loc="upper right")
    plt.tight_layout()
    plt.savefig("/kaggle/working/checkpoint_sizes.png", dpi=150, bbox_inches="tight")
    plt.show()